# KHÁM PHÁ DỮ LIỆU BRONZE (EDA) & BỘ TIỀN XỬ LÝ CHUẨN HÓA TẦNG SILVER
**Hệ thống:** Data Lakehouse Tổng hợp, Phân tích Thị trường Việc làm IT & Trợ lý AI  
**Nhóm sinh viên thực hiện:** Nguyễn Quân Bảo (23110181) & Nguyễn Hữu Thưởng (23110338)  

---

### Mục tiêu Thực Nghiệm:
1. **Khám phá Dữ liệu Thô (Bronze Layer Profiling):** Phân tích toàn bộ 8.064 tin tuyển dụng từ 4 nền tảng (ITViec, TopCV, TopDev, VietnamWorks). Đánh giá sự phân mảnh cấu trúc (102 trường dị thể) và ma trận khuyết thiếu (Missing Rate Matrix).
2. **Xây dựng Bộ Tiền Xử Lý Chuẩn Hóa (Silver Transformers):** Triển khai 7 modules làm sạch theo đúng thiết kế Canonical Schema trong tài liệu `tiền xử lý.md` (Lương, Kinh nghiệm, Địa phương, Cấp bậc, Tech Taxonomy, Thời gian).
3. **Thực thi Chuyển đổi Hợp nhất (Medallion Silver Layer):** Gom 4 nguồn dữ liệu dị thể về bảng chuẩn hóa duy nhất `silver_job_postings` với cơ chế bảo toàn thông tin gốc (`platform_specific_attributes`).
4. **Đánh giá Chất lượng Dữ liệu (Data Quality Scorecard):** So sánh tỷ lệ khả dụng trước và sau khi làm sạch.
5. **Trực quan hóa Thị trường (Market Insights):** Sinh 6 biểu đồ phân tích chuyên sâu (PNG độ phân giải cao).
6. **Lưu trữ & Kiểm soát Output:** Toàn bộ kết quả (Biểu đồ, Báo cáo JSON, Bảng CSV, File Parquet và Mẫu JSONL) được lưu trữ tập trung tại thư mục `notebook/output/`.


In [ ]:
# 1. Cấu hình Môi trường, Thư viện và Đường dẫn Linh hoạt
import os
import sys
import json
import re
import math
from datetime import datetime, timedelta
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình thẩm mỹ đồ họa
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# Thiết lập đường dẫn linh hoạt (tự động nhận diện dù chạy từ VS Code hay Terminal)
current_dir = os.getcwd()
if os.path.exists(os.path.join(current_dir, "data")):
    BASE_DIR = current_dir
elif os.path.exists(os.path.join(current_dir, "..", "data")):
    BASE_DIR = os.path.abspath(os.path.join(current_dir, ".."))
elif os.path.exists(r"d:\hcmute\TLCN\IT-Job-Data-Lakehouse\data"):
    BASE_DIR = r"d:\hcmute\TLCN\IT-Job-Data-Lakehouse"
else:
    BASE_DIR = current_dir

DATA_DIR = os.path.join(BASE_DIR, "data")
NOTEBOOK_DIR = os.path.join(BASE_DIR, "notebook")
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, "output")

# Đảm bảo toàn bộ artifacts xuất ra nằm gọn trong notebook/output/
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[*] Thư mục Gốc Dự Án (Base Dir):    {BASE_DIR}")
print(f"[*] Thư mục Dữ Liệu Thô (Data Dir):   {DATA_DIR}")
print(f"[*] Thư mục Xuất Kết Quả (Output):     {OUTPUT_DIR}")


## 1. Nạp Dữ Liệu Thô Tầng Bronze & Thống Kê Tổng Quan
Đọc toàn bộ 4 file `.jsonl` thô đã cào từ bot về bộ nhớ để tiến hành khảo sát và đánh giá sơ bộ.


In [ ]:
# 2. Đọc 4 file JSONL dữ liệu thô tầng Bronze
platforms = {
    'itviec': os.path.join(DATA_DIR, 'itviec_scraped_jobs.jsonl'),
    'topcv': os.path.join(DATA_DIR, 'topcv_scraped_jobs.jsonl'),
    'topdev': os.path.join(DATA_DIR, 'topdev_scraped_jobs.jsonl'),
    'vietnamworks': os.path.join(DATA_DIR, 'vietnamworks_scraped_jobs.jsonl')
}

raw_data = {}
summary_stats = []

for platform, file_path in platforms.items():
    if not os.path.exists(file_path):
        print(f"[CẢNH BÁO] Không tìm thấy file: {file_path}")
        continue
        
    records = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue
            try:
                records.append(json.loads(line_str))
            except Exception as e:
                print(f"[!] Lỗi parse JSON dòng {line_num} trong {platform}: {e}")
                
    raw_data[platform] = records
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    total_fields = len(records[0].keys()) if records else 0
    
    summary_stats.append({
        'Nền tảng': platform.upper(),
        'Số tin cào được': len(records),
        'Dung lượng (MB)': round(file_size_mb, 2),
        'Số trường thô': total_fields,
        'Trạng thái': 'Đã nạp thành công'
    })

df_raw_summary = pd.DataFrame(summary_stats)
print("=== THỐNG KÊ TỔNG QUAN TẦNG BRONZE ===")
display(df_raw_summary)
total_jobs = sum(s['Số tin cào được'] for s in summary_stats)
total_mb = sum(s['Dung lượng (MB)'] for s in summary_stats)
print(f"\n=> TỔNG CỘNG: {total_jobs:,} tin tuyển dụng IT | Tổng dung lượng thô: {total_mb:.2f} MB.")


## 2. Phân Tích Chất Lượng Dữ Liệu Thô (Missing Rate Matrix)
Mỗi nền tảng có cách đặt tên và tổ chức thuộc tính khác nhau (Heterogeneous Schema). Ta tiến hành lập ma trận kiểm tra tỷ lệ khuyết thiếu (Null/Missing Rate) đối với 7 nhóm thuộc tính cốt lõi của tin tuyển dụng.


In [ ]:
# 3. Tính toán Tỷ lệ Khuyết thiếu (Missing Rate) trên 7 trường cốt lõi
key_attributes = ['Lương (Salary)', 'Địa điểm (Location)', 'Kinh nghiệm (Exp)', 
                  'Kỹ năng (Skills)', 'Cấp bậc (Level)', 'Quy mô Cty (Size)', 'Ngành nghề (Industry)']

profiling_rows = []

for platform, records in raw_data.items():
    total = len(records)
    if total == 0:
        continue
        
    for attr in key_attributes:
        non_null_count = 0
        for r in records:
            val = None
            if attr == 'Lương (Salary)':
                val = r.get('salary')
            elif attr == 'Địa điểm (Location)':
                val = r.get('address') or r.get('job_location') or r.get('company_location') or r.get('work_address')
            elif attr == 'Kinh nghiệm (Exp)':
                val = r.get('experience') or r.get('years_of_experience') or r.get('skills_and_exp')
            elif attr == 'Kỹ năng (Skills)':
                val = r.get('skills') or r.get('required_skills') or r.get('overview_skills')
            elif attr == 'Cấp bậc (Level)':
                val = r.get('job_level') or r.get('job_function')
            elif attr == 'Quy mô Cty (Size)':
                val = r.get('company_size')
            elif attr == 'Ngành nghề (Industry)':
                val = r.get('company_industry') or r.get('job_industry')
                
            if val is not None and str(val).strip() != '' and str(val).strip().lower() not in ['none', 'null', 'nan']:
                non_null_count += 1
                
        missing_rate = ((total - non_null_count) / total) * 100
        profiling_rows.append({
            'Nền tảng': platform.upper(),
            'Thuộc tính': attr,
            'Số bản ghi có dữ liệu': non_null_count,
            'Tỷ lệ thiếu (%)': round(missing_rate, 2)
        })

df_profiling = pd.DataFrame(profiling_rows)
pivot_missing = df_profiling.pivot(index='Thuộc tính', columns='Nền tảng', values='Tỷ lệ thiếu (%)')
print("=== MA TRẬN TỶ LỆ KHUYẾT THIẾU DỮ LIỆU THÔ (%) ===")
display(pivot_missing)


In [ ]:
# 4. Trực quan hóa Ma trận Tỷ lệ Khuyết thiếu (Heatmap)
plt.figure(figsize=(10, 5))
sns.heatmap(pivot_missing, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': 'Tỷ lệ thiếu (%)'}, linewidths=0.5)
plt.title("MA TRẬN TỶ LỆ KHUYẾT THIẾU DỮ LIỆU THÔ (BRONZE LAYER)", fontsize=13, pad=15, fontweight='bold')
plt.ylabel("Thuộc tính quan trọng", fontweight='bold')
plt.xlabel("Nền tảng tuyển dụng", fontweight='bold')
plt.tight_layout()

heatmap_file = os.path.join(OUTPUT_DIR, "00_raw_missing_rate_heatmap.png")
plt.savefig(heatmap_file, dpi=150)
plt.show()
print(f"[OK] Đã lưu biểu đồ ma trận khuyết thiếu tại: {heatmap_file}")


## 3. Thiết Kế Bộ Tiền Xử Lý Chuẩn Hóa Tầng Silver (Silver Transformers)
Theo kiến trúc đã xác lập trong tài liệu `tiền xử lý.md`, ta xây dựng 7 module làm sạch chuyên biệt:
1. **Module 1 - Global ID & Audit:** `unified_job_id = {platform}_{original_job_id}`
2. **Module 2 - Salary Normalizer:** Phân rã khoảng lương, xử lý USD sang VNĐ (FX=25.400), phát hiện "Thương lượng/Thoả thuận".
3. **Module 3 - Experience Normalizer:** Chuẩn hóa số năm kinh nghiệm (`exp_years_min`, `exp_years_max`).
4. **Module 4 - City Normalizer:** Chuẩn hóa địa giới hành chính về 63 tỉnh thành & trung tâm IT chính.
5. **Module 5 - Level & Work Mode Normalizer:** Quy nạp cấp bậc (Intern, Fresher, Junior, Mid, Senior, Lead) và hình thức làm việc.
6. **Module 6 - Tech Stack Taxonomy:** Ánh xạ từ điển công nghệ > 120 kỹ năng IT chuẩn.
7. **Module 7 - Temporal Normalizer:** Quy đổi thời gian tương đối và đếm lùi về chuẩn ISO `YYYY-MM-DD`.


In [ ]:
# 5. [Module 2] Bộ Chuẩn Hóa Mức Lương (Salary Normalizer)
def parse_salary(salary_raw, fx_usd_vnd=25400):
    """
    Phân tích chuỗi lương thô đa định dạng từ 4 sàn về chuẩn số nguyên VNĐ hàng tháng.
    Trả về: (salary_min_vnd, salary_max_vnd, is_negotiable, currency)
    """
    if not salary_raw:
        return None, None, True, None
        
    raw = str(salary_raw).strip()
    lower = raw.lower()
    
    # 1. Nhận diện các trạng thái thương lượng / thỏa thuận / hấp dẫn
    negotiable_keywords = [
        'thoả thuận', 'thỏa thuận', 'thương lượng', 'negotiable', 
        'attractive', 'love it', 'cạnh tranh', 'thoa thuan', 'lương thoả thuận'
    ]
    if any(k in lower for k in negotiable_keywords):
        return None, None, True, None
        
    is_usd = ('usd' in lower) or ('$' in raw)
    is_yearly = ('năm' in lower) or ('year' in lower)
    
    # 2. Làm sạch dấu phẩy phân tách hàng nghìn: 1,400 -> 1400
    cleaned = re.sub(r'(\d),(\d{3})', r'\g<1>\g<2>', raw)
    # Loại bỏ dấu chấm phân tách hàng nghìn của VNĐ: 15.000.000 -> 15000000
    cleaned = re.sub(r'\.(?=\d{3}(?:\.|\D|$))', '', cleaned)
    
    # 3. Trích xuất danh sách các con số
    nums = [float(n) for n in re.findall(r'(\d+(?:\.\d+)?)', cleaned)]
    if not nums:
        return None, None, True, None
        
    # 4. Xác định hệ số nhân quy đổi về đơn vị VNĐ
    multiplier = 1
    if is_usd:
        multiplier = fx_usd_vnd
    elif any(k in lower for k in ['triệu', 'tr', 'm', 'vnd', '₫']):
        if max(nums) < 1000:
            multiplier = 1_000_000
        else:
            multiplier = 1
    elif max(nums) < 1000:
        multiplier = 1_000_000
        
    # Nếu là lương theo năm -> quy về tháng
    if is_yearly:
        multiplier /= 12.0
        
    min_sal = None
    max_sal = None
    
    # 5. Phân bổ cận dưới và cận trên
    if len(nums) >= 2:
        min_sal = int(nums[0] * multiplier)
        max_sal = int(nums[1] * multiplier)
    elif any(k in lower for k in ['từ', 'from', 'trên', 'over', 'min', 'ít nhất']):
        min_sal = int(nums[0] * multiplier)
    elif any(k in lower for k in ['tới', 'up to', 'đến', 'dưới', 'max', 'khoảng', 'approximately']):
        max_sal = int(nums[0] * multiplier)
    else:
        min_sal = int(nums[0] * multiplier)
        max_sal = int(nums[0] * multiplier)
        
    currency = 'USD' if is_usd else 'VND'
    return min_sal, max_sal, False, currency

# Kiểm thử các mẫu lương tiêu biểu của 4 sàn
test_salaries = [
    "You'll love it", "Thoả thuận", "Negotiable", "Thương lượng",
    "1,400 - 1,800 USD", "Approximately $230", "2,000 - 3,500 USD",
    "Từ 12 triệu", "9 - 30 triệu", "Tới 30 triệu", "500 - 1,200 USD",
    "15.000.000 VND to 30.000.000 VND", "Up to 50.000.000 VND",
    "14tr-16tr ₫/tháng", "$$ 3,000-5,000 /tháng", "Tới 50tr ₫/tháng", "Từ 185tr ₫/năm"
]

print("=== KIỂM THỬ THUẬT TOÁN PARSE LƯƠNG ===")
for ts in test_salaries:
    res = parse_salary(ts)
    print(f"{ts:35} => Min: {str(res[0]):>10} | Max: {str(res[1]):>10} | Thoả thuận: {str(res[2]):<5} | Ngoại tệ: {res[3]}")


In [ ]:
# 6. [Module 3] Bộ Chuẩn Hóa Số Năm Kinh Nghiệm (Experience Normalizer)
def parse_experience(exp_val, jd_text=""):
    """
    Phân tích chuỗi kinh nghiệm về khoảng năm (exp_years_min, exp_years_max).
    """
    raw = str(exp_val).strip() if exp_val else ""
    lower = raw.lower()
    
    # 1. Trường hợp không yêu cầu kinh nghiệm
    if any(k in lower for k in ['không yêu cầu', 'not required', 'chưa có kinh nghiệm', 'không cần']):
        return 0.0, 0.0
        
    # 2. Trường hợp dưới 1 năm
    if 'dưới 1 năm' in lower or 'under 1 year' in lower or '< 1' in lower:
        return 0.0, 1.0
        
    # 3. Tìm các con số trong chuỗi exp
    nums = [float(n) for n in re.findall(r'(\d+(?:\.\d+)?)', raw)]
    
    if len(nums) >= 2:
        return min(nums[0], nums[1]), max(nums[0], nums[1])
    elif len(nums) == 1:
        val = nums[0]
        if any(k in lower for k in ['trên', 'hơn', 'over', '+', 'tối thiểu', 'ít nhất', 'from']):
            return val, None
        elif any(k in lower for k in ['dưới', 'under', 'tối đa', 'up to']):
            return 0.0, val
        else:
            return val, val
            
    # 4. Fallback: Nếu trường exp rỗng (như ITViec), quét nhanh trong JD
    if jd_text:
        match = re.search(r'(\d+)\s*(?:-|đến|to)\s*(\d+)\s*(?:năm|year)', jd_text, re.IGNORECASE)
        if match:
            return float(match.group(1)), float(match.group(2))
        single_match = re.search(r'(?:ít nhất|tối thiểu|từ|over)\s*(\d+)\s*(?:năm|year)', jd_text, re.IGNORECASE)
        if single_match:
            return float(single_match.group(1)), None
            
    return None, None

# Kiểm thử
test_exps = ["Không yêu cầu", "Dưới 1 năm", "1 năm", "2 năm", "1 year, 3 years", "5 years+", "3", "0"]
print("=== KIỂM THỬ THUẬT TOÁN PARSE KINH NGHIỆM ===")
for te in test_exps:
    emin, emax = parse_experience(te)
    print(f"{te:25} => Min: {emin} năm | Max: {emax} năm")


In [ ]:
# 7. [Module 4] Bộ Chuẩn Hóa Tỉnh Thành & Địa Bàn (City Normalizer)
def parse_city(location_raw):
    """
    Quy chuẩn địa chỉ tự do về các đơn vị hành chính và trung tâm IT chính.
    """
    if not location_raw:
        return "Khác / Chưa rõ"
        
    text = str(location_raw).strip()
    lower = text.lower()
    
    # Hà Nội
    if any(k in lower for k in ['hà nội', 'ha noi', 'hanoi', 'hn', 'cầu giấy', 'nam từ liêm', 'bắc từ liêm', 'hoàn kiếm', 'ba đình', 'đống đa', 'thanh xuân', 'mễ trì', 'hai bà trưng']):
        return "Hà Nội"
        
    # TP. Hồ Chí Minh
    if any(k in lower for k in ['hồ chí minh', 'ho chi minh', 'hcm', 'tp.hcm', 'saigon', 'sài gòn', 'quận 1', 'quận 2', 'quận 3', 'quận 7', 'tân bình', 'thủ đức', 'bình thạnh', 'phú nhuận', 'kcx tân thuận']):
        return "TP. Hồ Chí Minh"
        
    # Đà Nẵng
    if any(k in lower for k in ['đà nẵng', 'da nang', 'danang']):
        return "Đà Nẵng"
        
    # Các tỉnh vệ tinh công nghiệp & công nghệ
    provinces = {
        'Cần Thơ': ['cần thơ', 'can tho'],
        'Hải Phòng': ['hải phòng', 'hai phong'],
        'Bắc Ninh': ['bắc ninh', 'bac ninh'],
        'Bình Dương': ['bình dương', 'binh duong'],
        'Đồng Nai': ['đồng nai', 'dong nai'],
        'Bà Rịa - Vũng Tàu': ['vũng tàu', 'vung tau', 'bà rịa'],
        'Huế': ['thừa thiên huế', 'huế', 'hue'],
        'Quảng Nam': ['quảng nam', 'quang nam']
    }
    
    for prov_name, aliases in provinces.items():
        if any(a in lower for a in aliases):
            return prov_name
            
    if 'remote' in lower or 'toàn quốc' in lower:
        return "Toàn quốc / Remote"
        
    return "Khác / Chưa rõ"

# Kiểm thử
test_locs = [
    "Tòa nhà MB, số 18 Lê Văn Lương, Phường Yên Hòa, Ha Noi",
    "364 Cong Hoa street, Tan Binh, Ho Chi Minh",
    "Quận Cầu Giấy, Hà Nội",
    "Thành phố Thủ Đức, Hồ Chí Minh",
    "Da Nang", "Bắc Ninh", "Làm việc từ xa (Remote)"
]
print("=== KIỂM THỬ THUẬT TOÁN PARSE ĐỊA ĐIỂM ===")
for tl in test_locs:
    print(f"{tl[:45]:45} => {parse_city(tl)}")


In [ ]:
# 8. [Module 5] Bộ Chuẩn Hóa Cấp Bậc & Hình Thức Làm Việc
def parse_job_level(level_raw, job_title=""):
    """
    Chuẩn hóa cấp bậc việc làm: Intern, Fresher, Junior, Middle, Senior, Lead/Manager, Director.
    """
    combined = f"{str(level_raw or '')} {str(job_title or '')}".lower()
    
    if any(k in combined for k in ['director', 'giám đốc', 'head of', 'cto', 'vp']):
        return "Director / Executive"
    if any(k in combined for k in ['manager', 'trưởng phòng', 'phó phòng', 'lead', 'trưởng nhóm', 'quản lý', 'supervisor']):
        return "Lead / Manager"
    if any(k in combined for k in ['senior', 'chuyên viên cấp cao', 'cấp cao', 'sr.', 'sr ']):
        return "Senior"
    if any(k in combined for k in ['middle', 'mid', 'chuyên viên']):
        return "Middle"
    if any(k in combined for k in ['junior', 'jr.', 'jr ']):
        return "Junior"
    if any(k in combined for k in ['fresher', 'entry level', 'mới tốt nghiệp']):
        return "Fresher"
    if any(k in combined for k in ['intern', 'thực tập', 'trainee', 'sinh viên']):
        return "Intern / Sinh viên"
        
    return "Nhân viên / Chưa phân cấp"

def parse_work_mode(mode_raw, text=""):
    """
    Chuẩn hóa hình thức: On-site, Hybrid, Remote.
    """
    combined = f"{str(mode_raw or '')} {str(text or '')}".lower()
    if 'hybrid' in combined:
        return "Hybrid"
    if any(k in combined for k in ['remote', 'từ xa', 'work from home', 'wfh']):
        return "Remote"
    if any(k in combined for k in ['on-site', 'onsite', 'tại văn phòng', 'toàn thời gian']):
        return "On-site"
    return "On-site"

def parse_vacancies(val):
    """
    Trích xuất số lượng tuyển dụng dạng nguyên dương.
    """
    if val is None or val == '':
        return 1
    m = re.search(r'\d+', str(val))
    return int(m.group(0)) if m else 1

def parse_int_safe(val):
    """
    Trích xuất số nguyên an toàn cho views_count và applications_count.
    """
    if val is None or val == '' or str(val).lower() in ['none', 'null', 'nan']:
        return None
    m = re.search(r'\d+', str(val))
    return int(m.group(0)) if m else None


In [ ]:
# 9. [Module 6] Từ Điển Kỹ Năng Chuẩn Hóa (Tech Stack Taxonomy)
TECH_TAXONOMY = {
    # 1. Ngôn ngữ lập trình
    'Python': ['python', 'py'],
    'Java': ['java', 'core java'],
    'JavaScript': ['javascript', 'js', 'es6'],
    'TypeScript': ['typescript', 'ts'],
    'C++': ['c++', 'cpp', 'c/c++'],
    'C#': ['c#', 'csharp', '.net c#'],
    'PHP': ['php'],
    'Go': ['golang', 'go lang', 'go'],
    'Ruby': ['ruby', 'ruby on rails', 'rails'],
    'Rust': ['rust'],
    'Swift': ['swift'],
    'Kotlin': ['kotlin'],
    'Dart': ['dart'],
    'Scala': ['scala'],
    'SQL': ['sql', 't-sql', 'pl/sql'],
    
    # 2. Frontend
    'React': ['react', 'reactjs', 'react.js'],
    'Angular': ['angular', 'angularjs', 'angular.js'],
    'Vue.js': ['vue', 'vuejs', 'vue.js'],
    'Next.js': ['nextjs', 'next.js'],
    'HTML/CSS': ['html', 'html5', 'css', 'css3', 'tailwind', 'bootstrap'],
    
    # 3. Backend & Frameworks
    'Node.js': ['nodejs', 'node.js', 'node'],
    'Spring Boot': ['spring boot', 'springboot', 'spring framework', 'spring'],
    'Django': ['django'],
    'FastAPI': ['fastapi'],
    'Flask': ['flask'],
    '.NET': ['.net', 'asp.net', 'dotnet', '.net core'],
    'Laravel': ['laravel'],
    'Microservices / REST API': ['microservices', 'restful api', 'rest api'],
    
    # 4. Mobile
    'Flutter': ['flutter'],
    'React Native': ['react native', 'reactnative'],
    'iOS': ['ios'],
    'Android': ['android'],
    
    # 5. Cơ sở dữ liệu
    'PostgreSQL': ['postgresql', 'postgres'],
    'MySQL': ['mysql'],
    'MongoDB': ['mongodb', 'mongo'],
    'Redis': ['redis'],
    'Oracle': ['oracle', 'oracle db'],
    'SQL Server': ['sql server', 'mssql'],
    'Elasticsearch': ['elasticsearch', 'elastic search'],
    
    # 6. Cloud & DevOps
    'AWS': ['aws', 'amazon web services'],
    'Azure': ['azure', 'microsoft azure'],
    'GCP': ['gcp', 'google cloud'],
    'Docker': ['docker'],
    'Kubernetes': ['kubernetes', 'k8s'],
    'CI/CD': ['ci/cd', 'cicd', 'jenkins', 'gitlab'],
    'Linux': ['linux', 'ubuntu'],
    'Git': ['git', 'github'],
    
    # 7. Data Engineering & AI
    'Apache Spark': ['spark', 'pyspark', 'apache spark'],
    'Kafka': ['kafka', 'apache kafka'],
    'Airflow': ['airflow', 'apache airflow'],
    'Big Data': ['big data', 'hadoop'],
    'PyTorch': ['pytorch'],
    'TensorFlow': ['tensorflow'],
    'Machine Learning / AI': ['machine learning', 'deep learning', 'ai', 'data science', 'nlp', 'llm', 'computer vision'],
    
    # 8. IT Roles & Specialties
    'QA / QC / Tester': ['tester', 'qa', 'qc', 'quality control', 'automation test', 'manual test', 'selenium'],
    'Business Analyst (BA)': ['business analyst', 'ba', 'product owner', 'scrum master'],
    'Cybersecurity': ['it security', 'cyber security', 'information security', 'security assessment', 'soc'],
    'UI/UX Design': ['ui/ux', 'ui / ux', 'figma'],
    'System / Network Engineer': ['system admin', 'network engineer', 'ccna', 'ccnp', 'helpdesk']
}

# Tiền biên dịch regex patterns để tăng tốc tối đa
COMPILED_TAXONOMY = {}
for canonical_name, aliases in TECH_TAXONOMY.items():
    pats = []
    for alias in aliases:
        # Regex boundary an toàn: hỗ trợ cả ký tự đặc biệt như c++, c#, .net
        pat = re.compile(r'(?<![a-zA-Z0-9])' + re.escape(alias.lower()) + r'(?![a-zA-Z0-9])', re.IGNORECASE)
        pats.append(pat)
    COMPILED_TAXONOMY[canonical_name] = pats

def standardize_skills(raw_skills, job_title="", text=""):
    """
    Bóc tách và quy chuẩn danh mục kỹ năng từ trường raw skills kết hợp title và text.
    Trả về mảng các kỹ năng chuẩn hóa không trùng lặp (ARRAY<STRING>).
    """
    corpus = f"{str(raw_skills or '')} {str(job_title or '')}".lower()
    detected_skills = set()
    
    for canonical_name, pats in COMPILED_TAXONOMY.items():
        if any(p.search(corpus) for p in pats):
            detected_skills.add(canonical_name)
            
    # Fallback: nếu chưa bắt được kỹ năng nào từ title & raw_skills, quét thêm 400 ký tự đầu của JD
    if not detected_skills and text:
        snippet = text[:400].lower()
        for canonical_name, pats in COMPILED_TAXONOMY.items():
            if any(p.search(snippet) for p in pats):
                detected_skills.add(canonical_name)
                
    return sorted(list(detected_skills))

# Kiểm thử
sample_raw_skill = "Kỹ năng: ReactJS, node.js, aws, Python Programming, Docker and k8s, Spring Boot"
print("=== KIỂM THỬ CHUẨN HÓA KỸ NĂNG ===")
print("Chuỗi thô: ", sample_raw_skill)
print("Chuẩn hóa: ", standardize_skills(sample_raw_skill))


In [ ]:
# 10. [Module 7] Bộ Chuẩn Hóa Chu Kỳ Tuyển Dụng (Temporal Normalizer)
def parse_dates(crawled_at, posted_raw=None, deadline_raw=None, expires_in_raw=None, posted_ago_raw=None):
    """
    Quy đổi các mốc thời gian phân mảnh về chuẩn ISO YYYY-MM-DD.
    """
    # 1. Chuẩn hóa crawled_at
    crawled_date = None
    if crawled_at:
        try:
            crawled_date = datetime.strptime(str(crawled_at)[:10], "%Y-%m-%d").date()
        except:
            crawled_date = datetime.now().date()
    else:
        crawled_date = datetime.now().date()
        
    posted_date = None
    expired_date = None
    
    # 2. Xử lý posted_date
    if posted_raw:
        try:
            posted_date = datetime.strptime(str(posted_raw)[:10], "%Y-%m-%d").date()
        except:
            pass
            
    if not posted_date and posted_ago_raw:
        # VietnamWorks: "3 ngày trước" / "Đăng 5 ngày trước"
        num_match = re.search(r'(\d+)\s*(?:ngày|day)', str(posted_ago_raw), re.IGNORECASE)
        if num_match:
            days = int(num_match.group(1))
            posted_date = crawled_date - timedelta(days=days)
            
    if not posted_date:
        posted_date = crawled_date
        
    # 3. Xử lý expired_date
    if deadline_raw:
        try:
            # TopCV thường có DD/MM/YYYY
            if '/' in str(deadline_raw):
                expired_date = datetime.strptime(str(deadline_raw).strip(), "%d/%m/%Y").date()
            else:
                expired_date = datetime.strptime(str(deadline_raw)[:10], "%Y-%m-%d").date()
        except:
            pass
            
    if not expired_date and expires_in_raw:
        # TopDev: "còn 15 ngày" / "15 ngày nữa"
        num_match = re.search(r'(\d+)\s*(?:ngày|day)', str(expires_in_raw), re.IGNORECASE)
        if num_match:
            days = int(num_match.group(1))
            expired_date = crawled_date + timedelta(days=days)
            
    return crawled_date, posted_date, expired_date


## 4. Thực Thi Pipeline Chuẩn Hóa Toàn Diện (Bronze -> Silver)
Áp dụng toàn bộ 7 Transformers lên **8.064 bản ghi thô** từ 4 nền tảng. Các trường đặc thù không nằm trong Silver Schema sẽ được đóng gói thành JSON String trong cột `platform_specific_attributes` nhằm đảm bảo tính toàn vẹn (Zero Data Loss).


In [ ]:
# 11. Chạy chuyển đổi đồng loạt toàn bộ dữ liệu 4 sàn
silver_records = []
now_ts = datetime.now()

for platform_name, records in raw_data.items():
    print(f"[*] Đang xử lý nền tảng: {platform_name.upper()} ({len(records):,} bản ghi)...")
    
    for r in records:
        orig_id = str(r.get('job_id', '')).strip()
        unified_id = f"{platform_name}_{orig_id}"
        
        # 1. Parse Lương
        raw_sal = r.get('salary')
        s_min, s_max, is_neg, curr = parse_salary(raw_sal)
        
        # 2. Parse Địa điểm
        raw_addr = r.get('address') or r.get('job_location') or r.get('company_location') or r.get('work_address') or ""
        city = parse_city(raw_addr)
        
        # 3. Parse Kinh nghiệm
        raw_exp = r.get('experience') or r.get('years_of_experience') or r.get('skills_and_exp')
        e_min, e_max = parse_experience(raw_exp, jd_text=str(r.get('job_description', '')))
        
        # 4. Parse Cấp bậc & Hình thức làm việc
        raw_lvl = r.get('job_level') or r.get('job_function')
        job_title = str(r.get('job_title', '')).strip()
        level = parse_job_level(raw_lvl, job_title)
        
        raw_mode = r.get('work_mode')
        work_mode = parse_work_mode(raw_mode, job_title)
        
        # 5. Parse Kỹ năng
        raw_skills = r.get('skills') or r.get('required_skills') or r.get('overview_skills') or ""
        skills_std = standardize_skills(raw_skills, job_title, str(r.get('job_description', '')))
        
        # 6. Parse Ngày tháng
        crawled_dt, posted_dt, expired_dt = parse_dates(
            crawled_at=r.get('crawled_date') or r.get('scraped_at'),
            posted_raw=r.get('posted_date'),
            deadline_raw=r.get('deadline') or r.get('expired_on'),
            expires_in_raw=r.get('expires_in'),
            posted_ago_raw=r.get('posted_time_ago')
        )
        
        # 7. Đóng gói trường đặc thù
        consumed_keys = {'platform', 'job_id', 'job_title', 'salary', 'job_url', 'content_hash', 
                         'crawled_date', 'scraped_at', 'address', 'job_location', 'company_location',
                         'work_address', 'experience', 'years_of_experience', 'skills_and_exp',
                         'skills', 'required_skills', 'overview_skills', 'job_level', 'job_function',
                         'work_mode', 'posted_date', 'deadline', 'expired_on', 'expires_in', 'posted_time_ago'}
        
        specific_attrs = {k: v for k, v in r.items() if k not in consumed_keys and v is not None}
        
        silver_rec = {
            'unified_job_id': unified_id,
            'platform': platform_name,
            'original_job_id': orig_id,
            'job_url': r.get('job_url'),
            'content_hash': r.get('content_hash'),
            'crawled_at': str(crawled_dt),
            'silver_updated_at': str(now_ts),
            'is_active': True,
            
            'job_title': job_title,
            'job_level': level,
            'job_expertise': r.get('job_expertise'),
            'job_domain': r.get('job_domain'),
            'work_mode': work_mode,
            'working_time': r.get('working_days') or r.get('work_time'),
            'overtime_policy': r.get('overtime_policy'),
            
            'company_name': str(r.get('company_name', '')).strip(),
            'company_size': r.get('company_size'),
            'company_industry': r.get('company_industry') or r.get('job_industry'),
            'company_type': r.get('company_type'),
            'company_country': r.get('country') or r.get('company_country') or r.get('nationality'),
            
            'city': city,
            'full_address': raw_addr,
            
            'salary_raw': raw_sal,
            'salary_min_vnd': s_min,
            'salary_max_vnd': s_max,
            'is_negotiable': is_neg,
            'salary_currency': curr,
            'exp_years_min': e_min,
            'exp_years_max': e_max,
            'vacancies_count': parse_vacancies(r.get('quantity')),
            
            'views_count': parse_int_safe(r.get('views_count')),
            'applications_count': parse_int_safe(r.get('num_candidates') or r.get('applicant_count')),
            
            'posted_date': str(posted_dt) if posted_dt else None,
            'expired_date': str(expired_dt) if expired_dt else None,
            
            'skills_raw': [s.strip() for s in str(raw_skills).split(',') if s.strip()] if raw_skills else [],
            'skills_standardized': skills_std,
            'job_description_clean': str(r.get('job_description') or r.get('general_description') or ''),
            'requirements_clean': str(r.get('candidate_requirements') or r.get('requirements') or r.get('job_requirements') or ''),
            'benefits_clean': str(r.get('benefits') or r.get('why_love_working_here') or r.get('what_we_offer') or ''),
            
            'platform_specific_attributes': json.dumps(specific_attrs, ensure_ascii=False)
        }
        silver_records.append(silver_rec)

df_silver = pd.DataFrame(silver_records)

# Ép kiểu dữ liệu chuẩn (Strong Typing) phục vụ Parquet và Spark
df_silver['salary_min_vnd'] = pd.to_numeric(df_silver['salary_min_vnd'], errors='coerce').astype('Int64')
df_silver['salary_max_vnd'] = pd.to_numeric(df_silver['salary_max_vnd'], errors='coerce').astype('Int64')
df_silver['views_count'] = pd.to_numeric(df_silver['views_count'], errors='coerce').astype('Int64')
df_silver['applications_count'] = pd.to_numeric(df_silver['applications_count'], errors='coerce').astype('Int64')
df_silver['vacancies_count'] = pd.to_numeric(df_silver['vacancies_count'], errors='coerce').astype('int64')

print(f"\n[THÀNH CÔNG] Đã chuyển đổi hoàn tất {len(df_silver):,} bản ghi lên Canonical Schema Tầng Silver!")
print(f"[*] Kích thước DataFrame Silver: {df_silver.shape[0]:,} dòng x {df_silver.shape[1]} cột.")


## 5. Đánh Giá Chất Lượng Dữ Liệu Tầng Silver (Data Quality Scorecard)
Kiểm tra đối soát chất lượng trước và sau khi làm sạch:
- Tỷ lệ tin có thông tin mức lương rõ ràng (đã parse được khoảng số hoặc đánh dấu Thoả thuận).
- Tỷ lệ chuẩn hóa tỉnh thành thành công.
- Tỷ lệ trích xuất kỹ năng công nghệ.
- Tỷ lệ chuẩn hóa cấp bậc.


In [ ]:
# 12. Bảng Đối Soát Chất Lượng Before vs After
total_silver = len(df_silver)
valid_salary = df_silver['salary_min_vnd'].notnull() | df_silver['salary_max_vnd'].notnull() | df_silver['is_negotiable']
valid_city = (df_silver['city'].notnull()) & (df_silver['city'] != 'Khác / Chưa rõ')
valid_skills = df_silver['skills_standardized'].apply(lambda x: len(x) > 0)
valid_exp = df_silver['exp_years_min'].notnull() | df_silver['exp_years_max'].notnull()
valid_level = df_silver['job_level'].notnull() & (df_silver['job_level'] != 'Nhân viên / Chưa phân cấp')

quality_metrics = [
    {
        'Thuộc tính': 'Mức Lương (Salary Info)',
        'Tiêu chí chất lượng': 'Có khoảng số VNĐ hoặc cờ Thoả thuận',
        'Số lượng đạt chuẩn': int(valid_salary.sum()),
        'Tỷ lệ khả dụng (%)': round((valid_salary.sum() / total_silver) * 100, 2)
    },
    {
        'Thuộc tính': 'Địa điểm (Standardized City)',
        'Tiêu chí chất lượng': 'Quy chuẩn về tỉnh thành/trung tâm IT rõ ràng',
        'Số lượng đạt chuẩn': int(valid_city.sum()),
        'Tỷ lệ khả dụng (%)': round((valid_city.sum() / total_silver) * 100, 2)
    },
    {
        'Thuộc tính': 'Kỹ năng Công nghệ (Tech Skills)',
        'Tiêu chí chất lượng': 'Trích xuất ít nhất 1 kỹ năng chuẩn Taxonomy',
        'Số lượng đạt chuẩn': int(valid_skills.sum()),
        'Tỷ lệ khả dụng (%)': round((valid_skills.sum() / total_silver) * 100, 2)
    },
    {
        'Thuộc tính': 'Kinh nghiệm (Experience Range)',
        'Tiêu chí chất lượng': 'Có số năm min hoặc max cụ thể',
        'Số lượng đạt chuẩn': int(valid_exp.sum()),
        'Tỷ lệ khả dụng (%)': round((valid_exp.sum() / total_silver) * 100, 2)
    },
    {
        'Thuộc tính': 'Cấp bậc (Job Level)',
        'Tiêu chí chất lượng': 'Phân cấp rõ (Intern, Fresher, Jr, Mid, Sr, Lead)',
        'Số lượng đạt chuẩn': int(valid_level.sum()),
        'Tỷ lệ khả dụng (%)': round((valid_level.sum() / total_silver) * 100, 2)
    }
]

df_scorecard = pd.DataFrame(quality_metrics)
print("=== BẢNG ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU TẦNG SILVER (DATA QUALITY SCORECARD) ===")
display(df_scorecard)

# Lưu scorecard vào output
scorecard_csv = os.path.join(OUTPUT_DIR, "eda_quality_scorecard.csv")
df_scorecard.to_csv(scorecard_csv, index=False, encoding='utf-8-sig')
print(f"[OK] Đã lưu Scorecard tại: {scorecard_csv}")


## 6. Trực Quan Hóa Thị Trường Việc Làm IT (Market Insights)
Sinh và lưu 5 biểu đồ phân tích chuyên sâu chất lượng cao (High-resolution PNG, DPI=150) trực tiếp vào thư mục `notebook/output/`:
1. `01_platform_distribution.png`: Tỷ trọng tin cào theo 4 nền tảng.
2. `02_salary_distribution_vnd.png`: Phân bố mức lương trung bình IT theo triệu VNĐ.
3. `03_top_25_tech_skills.png`: Top 25 công nghệ / ngôn ngữ lập trình được săn đón nhất.
4. `04_job_level_distribution.png`: Cơ cấu việc làm theo cấp bậc kinh nghiệm.
5. `05_city_job_distribution.png`: Phân bố địa lý việc làm IT trên toàn quốc.


In [ ]:
# 13. [Biểu đồ 1] Phân bố số lượng tin theo nền tảng
plt.figure(figsize=(8, 4.5))
ax = sns.countplot(data=df_silver, x='platform', palette='viridis', order=df_silver['platform'].value_counts().index)
plt.title("PHÂN BỐ TIN TUYỂN DỤNG THEO NỀN TẢNG (BRONZE -> SILVER)", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Nền tảng", fontweight='bold')
plt.ylabel("Số lượng tin", fontweight='bold')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')
plt.tight_layout()
chart1_file = os.path.join(OUTPUT_DIR, "01_platform_distribution.png")
plt.savefig(chart1_file, dpi=150)
plt.show()

# 14. [Biểu đồ 2] Phân bố mức lương IT theo Triệu VNĐ
df_salary_plot = df_silver.dropna(subset=['salary_min_vnd', 'salary_max_vnd'], how='all').copy()
df_salary_plot['salary_avg_vnd'] = df_salary_plot[['salary_min_vnd', 'salary_max_vnd']].mean(axis=1)
df_salary_plot['salary_avg_mil'] = df_salary_plot['salary_avg_vnd'] / 1_000_000
df_salary_plot = df_salary_plot[(df_salary_plot['salary_avg_mil'] >= 3) & (df_salary_plot['salary_avg_mil'] <= 150)]

plt.figure(figsize=(9, 4.5))
sns.histplot(df_salary_plot['salary_avg_mil'], bins=35, kde=True, color='#2b5c8f')
median_sal = df_salary_plot['salary_avg_mil'].median()
plt.axvline(median_sal, color='red', linestyle='--', linewidth=2, label=f'Trung vị: {median_sal:.1f} triệu')
plt.title("PHÂN BỐ MỨC LƯƠNG TRUNG BÌNH THỊ TRƯỜNG IT (TRIỆU VNĐ / THÁNG)", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Mức lương trung bình (Triệu VNĐ)", fontweight='bold')
plt.ylabel("Số lượng vị trí", fontweight='bold')
plt.legend()
plt.tight_layout()
chart2_file = os.path.join(OUTPUT_DIR, "02_salary_distribution_vnd.png")
plt.savefig(chart2_file, dpi=150)
plt.show()

# 15. [Biểu đồ 3] Top 25 Kỹ năng Công nghệ Hot nhất
all_skills = []
for skills_list in df_silver['skills_standardized']:
    all_skills.extend(skills_list)
skill_counts = Counter(all_skills).most_common(25)
df_top_skills = pd.DataFrame(skill_counts, columns=['Skill', 'Count'])

plt.figure(figsize=(10, 7))
sns.barplot(data=df_top_skills, y='Skill', x='Count', palette='crest_r')
plt.title("TOP 25 KỸ NĂNG CÔNG NGHỆ ĐƯỢC TUYỂN DỤNG NHIỀU NHẤT", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Số lượng tin tuyển dụng yêu cầu", fontweight='bold')
plt.ylabel("Kỹ năng / Công nghệ", fontweight='bold')
for i, v in enumerate(df_top_skills['Count']):
    plt.text(v + 15, i, f"{v:,}", va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
chart3_file = os.path.join(OUTPUT_DIR, "03_top_25_tech_skills.png")
plt.savefig(chart3_file, dpi=150)
plt.show()

# 16. [Biểu đồ 4] Cơ cấu Cấp bậc Tuyển dụng
plt.figure(figsize=(8, 4.5))
order_levels = ['Intern / Sinh viên', 'Fresher', 'Junior', 'Middle', 'Senior', 'Lead / Manager', 'Director / Executive', 'Nhân viên / Chưa phân cấp']
order_levels = [l for l in order_levels if l in df_silver['job_level'].values]
ax = sns.countplot(data=df_silver, y='job_level', order=order_levels, palette='mako')
plt.title("CƠ CẤU VIỆC LÀM THEO CẤP BẬC KINH NGHIỆM", fontsize=13, pad=12, fontweight='bold')
plt.xlabel("Số lượng tin", fontweight='bold')
plt.ylabel("Cấp bậc", fontweight='bold')
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{int(width):,}", (width, p.get_y() + p.get_height() / 2.),
                xytext=(5, 0), textcoords='offset points', va='center', fontweight='bold')
plt.tight_layout()
chart4_file = os.path.join(OUTPUT_DIR, "04_job_level_distribution.png")
plt.savefig(chart4_file, dpi=150)
plt.show()

# 17. [Biểu đồ 5] Phân bố Địa lý Thị trường Việc làm IT
top_cities = df_silver['city'].value_counts().head(7)
plt.figure(figsize=(7, 7))
plt.pie(top_cities, labels=top_cities.index, autopct='%1.1f%%', startangle=140, 
        colors=sns.color_palette('pastel', len(top_cities)), explode=[0.05 if i < 2 else 0 for i in range(len(top_cities))])
plt.title("TỶ TRỌNG VIỆC LÀM IT THEO TỈNH THÀNH TOÀN QUỐC", fontsize=13, pad=12, fontweight='bold')
plt.tight_layout()
chart5_file = os.path.join(OUTPUT_DIR, "05_city_job_distribution.png")
plt.savefig(chart5_file, dpi=150)
plt.show()

print("[OK] Đã xuất thành công 5 biểu đồ trực quan hóa vào notebook/output/.")


## 7. Xuất Kết Quả Dữ Liệu Tầng Silver & Báo Cáo EDA (Artifacts Export)
Toàn bộ kết quả được xuất gọn gàng trong thư mục `notebook/output/`:
1. `silver_job_postings_sample.parquet`: Định dạng Parquet chuẩn nạp vào PySpark / Delta Lake.
2. `silver_job_postings_sample.jsonl`: 1.000 dòng mẫu JSON Lines để người dùng có thể mở kiểm tra trực tiếp.
3. `eda_summary_report.json`: Báo cáo tóm tắt các chỉ số vận hành và chất lượng.


In [ ]:
# 18. Xuất Dữ liệu mẫu Tầng Silver và Báo cáo Tổng kết
parquet_file = os.path.join(OUTPUT_DIR, "silver_job_postings_sample.parquet")
jsonl_file = os.path.join(OUTPUT_DIR, "silver_job_postings_sample.jsonl")
report_json_file = os.path.join(OUTPUT_DIR, "eda_summary_report.json")

# 1. Lưu Parquet (chuẩn bị nạp PySpark Delta Lake)
df_silver.to_parquet(parquet_file, index=False, engine='pyarrow')
print(f"[OK] Đã lưu Silver Parquet: {parquet_file} ({round(os.path.getsize(parquet_file)/(1024*1024), 2)} MB)")

# 2. Lưu mẫu 1.000 dòng JSON Lines để xem trực tiếp
sample_records = silver_records[:1000]
with open(jsonl_file, 'w', encoding='utf-8') as f:
    for rec in sample_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"[OK] Đã lưu mẫu 1.000 dòng JSON Lines: {jsonl_file}")

# 3. Lưu Báo cáo JSON tóm tắt
summary_report = {
    "project": "IT Job Data Lakehouse - Silver Layer EDA & Transformation",
    "execution_time": datetime.now().isoformat(),
    "total_raw_jobs": total_jobs,
    "total_silver_jobs": len(df_silver),
    "platform_breakdown": df_silver['platform'].value_counts().to_dict(),
    "city_breakdown": df_silver['city'].value_counts().head(5).to_dict(),
    "top_10_skills": [s[0] for s in skill_counts[:10]],
    "quality_metrics": quality_metrics
}
with open(report_json_file, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, ensure_ascii=False, indent=2)
print(f"[OK] Đã lưu Báo cáo JSON: {report_json_file}")

# 4. Liệt kê toàn bộ file artifacts trong thư mục notebook/output/
print("\n=== DANH SÁCH TOÀN BỘ FILE TRONG THƯ MỤC OUTPUT ===")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    fsize = os.path.getsize(fpath) / 1024
    print(f"- {fname:<35} | {fsize:>8.1f} KB")
